# DPDP adversarial stress test on Kaggle

Settings: GPU T4 x2, Internet on. Attach a dataset containing this repository.
Run with *Save Version → Save & Run All (Commit)* so the outputs are kept.

In [ ]:
import os, glob, shutil, subprocess, time
src = glob.glob('/kaggle/input/**/src/run_experiment.py', recursive=True)
REPO = os.path.dirname(os.path.dirname(src[0]))
WORK = '/kaggle/working/dpdp'
shutil.copytree(REPO, WORK, dirs_exist_ok=True, ignore=shutil.ignore_patterns('dpdp_db*', 'results', 'archive'))
print(sorted(os.listdir(f'{WORK}/data')))

In [ ]:
!apt-get install -y -qq zstd > /dev/null 2>&1 || true
!curl -fsSL https://ollama.com/install.sh | sh
assert shutil.which('ollama')
!pip install -q -r {WORK}/requirements.txt
env = dict(os.environ, OLLAMA_NUM_PARALLEL='4', OLLAMA_MAX_LOADED_MODELS='2')
subprocess.Popen(['ollama', 'serve'], env=env, stdout=open('/kaggle/working/ollama.log', 'w'), stderr=subprocess.STDOUT)
time.sleep(10)
for m in ['llama3.1:8b', 'qwen2.5:7b', 'gemma2:9b']:
    !ollama pull {m} > /dev/null
!ollama list

In [ ]:
OUT = '/kaggle/working/results'
RUN = f'cd {WORK} && python src/run_experiment.py --attacker-model qwen2.5:7b --workers 4 --temps 0.5 0.7 0.9'
!{RUN} --model llama3.1:8b --runs 3 --out {OUT}/llama3.1-8b
!{RUN} --model qwen2.5:7b --runs 1 --out {OUT}/qwen2.5-7b
!{RUN} --model gemma2:9b --runs 1 --out {OUT}/gemma2-9b

In [ ]:
!cd {OUT} && python {WORK}/src/analysis.py llama3.1-8b qwen2.5-7b gemma2-9b --out significance.json
!cd /kaggle/working && zip -qr results.zip results